# GVA2035_1_데이터수집복원

시나리오별 산업 실질부가가치 전망(2025~2035) 개발 코드. API 키(API_KEY_BOK.txt, API_KEY_KOSIS.txt) 필요.
저장소: https://github.com/sdkparkforbi/gva-scenario-2025-2035

## 1. fetch_gdp_quarterly.py — GDP 분기(계절조정) 수집

In [ ]:
# -*- coding: utf-8 -*-
"""한국은행 ECOS API로 계절조정 분기 GDP 시계열을 받아온다.

확보 시계열 (1960Q1 ~ 최신, 단위: 십억원 / 디플레이터는 지수):
  - 명목 GDP        : 200Y107 (국내총생산에 대한 지출, 계절조정, 명목, 분기)
  - 실질 GDP        : 200Y108 (국내총생산에 대한 지출, 계절조정, 실질, 분기)
  - GDP 디플레이터  : 명목/실질 x 100 으로 산출 (계절조정 시계열 일관 / 2020=100)
  항목코드 10601 = 국내총생산에 대한 지출(= GDP 총계)
"""
import csv
import requests

API_KEY = open("API_KEY_BOK.txt", encoding="utf-8").read().strip()
BASE = "http://ecos.bok.or.kr/api/StatisticSearch"
GDP_ITEM = "10601"
START, END = "1900Q1", "2099Q4"  # API 조회 범위 (실제 보유분만 반환)
# 2026Q1 계절조정 명목 GDP는 비정상값(직전 분기 대비 +10.5%, 디플레이터 급등)이라 제외.
# 신뢰 가능한 최신 분기까지만 사용.
CUTOFF = "2025Q4"
# 산업별 균형 패널(36개 말단 산업이 모두 존재하는 첫 분기)과 분석 기간을 일치.
START_USE = "1980Q1"


def fetch(table):
    url = f"{BASE}/{API_KEY}/json/kr/1/9999/{table}/Q/{START}/{END}/{GDP_ITEM}"
    rows = requests.get(url, timeout=60).json().get("StatisticSearch", {}).get("row", [])
    return {r["TIME"]: float(r["DATA_VALUE"]) for r in rows}


def build():
    nominal = fetch("200Y107")   # 계절조정 명목
    real = fetch("200Y108")      # 계절조정 실질
    rows = []
    for t in sorted(nominal):
        if t > CUTOFF or t < START_USE:
            continue
        nom, rl = nominal[t], real.get(t)
        defl = round(nom / rl * 100, 4) if rl else None  # GDP 디플레이터(2020=100)
        rows.append((t, nom, rl, defl))
    return rows


if __name__ == "__main__":
    rows = build()
    with open("gdp_quarterly_sa.csv", "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(["분기", "명목GDP_십억원", "실질GDP_십억원", "GDP디플레이터_2020=100"])
        w.writerows(rows)
    print(f"{len(rows)}개 분기 저장: gdp_quarterly_sa.csv ({rows[0][0]} ~ {rows[-1][0]})")
    print("\n최근 8분기:")
    print(f"{'분기':<8}{'명목GDP':>14}{'실질GDP':>14}{'디플레이터':>12}")
    for t, nom, rl, defl in rows[-8:]:
        print(f"{t:<8}{nom:>14,.1f}{rl:>14,.1f}{defl:>12,.2f}")


## 2. fetch_gdp_industry.py — 36개 산업 실질·명목 부가가치 수집

In [ ]:
# -*- coding: utf-8 -*-
"""한국은행 ECOS API로 경제활동별(산업별) GDP를 가장 세부 단위로 받아온다.

- 통계표: 200Y103 = 경제활동별 GDP 및 GNI(계절조정, 명목, 분기)
          200Y104 = 경제활동별 GDP 및 GNI(계절조정, 실질, 분기)
- 가장 작은 세부 단위(leaf) 36개 산업만 추출 (상위 집계·GDP/GNI 등 소득 항목 제외).
- 단위: 십억원. 기간: API 보유 전 구간(분기 GDP와 동일하게 1960Q1~ 가능 범위).
- 산출물:
    gdp_industry_long.csv : 분기 x 산업 long 포맷 (분기,산업코드,산업명,명목_십억원,실질_십억원)
    gdp_industry_coverage.csv : 산업별 데이터 보유 구간(시작/종료/개수)
"""
import csv
import requests

API_KEY = open("API_KEY_BOK.txt", encoding="utf-8").read().strip()
ITEM_URL = "http://ecos.bok.or.kr/api/StatisticItemList/{k}/json/kr/1/500/{t}"
DATA_URL = "http://ecos.bok.or.kr/api/StatisticSearch/{k}/json/kr/1/30000/{t}/Q/1900Q1/2099Q4"
NOM_TBL, REAL_TBL = "200Y103", "200Y104"
# 산업이 아닌 집계·소득 항목 (제외)
AGG = {"1200", "1300", "1400", "1500", "1600", "1700", "1800"}
CUTOFF = "2025Q4"  # 분기 GDP 시계열과 동일하게 최신 신뢰 분기까지만 사용 (2026Q1 제외)
START = "1980Q1"   # 36개 말단 산업이 모두 존재하는 첫 분기 → 결측 없는 균형 패널


def leaf_industries():
    """200Y103 항목 계층에서 말단(자식 없는) 산업 코드만 반환. 표시 순서 유지."""
    rows = requests.get(ITEM_URL.format(k=API_KEY, t=NOM_TBL), timeout=30) \
        .json().get("StatisticItemList", {}).get("row", [])
    order, name, parents = [], {}, set()
    for r in rows:
        ic = r["ITEM_CODE"]
        if ic not in name:
            order.append(ic)
            name[ic] = r["ITEM_NAME"]
        if r.get("P_ITEM_CODE"):
            parents.add(r["P_ITEM_CODE"])
    return [(ic, name[ic]) for ic in order if ic not in parents and ic not in AGG]


def fetch_all(table):
    """표 전체를 {item_code: {TIME: value}} 로 반환."""
    rows = requests.get(DATA_URL.format(k=API_KEY, t=table), timeout=120) \
        .json().get("StatisticSearch", {}).get("row", [])
    out = {}
    for r in rows:
        v = r.get("DATA_VALUE")
        if v in (None, "", "-") or r["TIME"] > CUTOFF or r["TIME"] < START:
            continue
        out.setdefault(r["ITEM_CODE1"], {})[r["TIME"]] = float(v)
    return out


def main():
    leaves = leaf_industries()
    nom, real = fetch_all(NOM_TBL), fetch_all(REAL_TBL)

    # long 포맷 저장
    with open("gdp_industry_long.csv", "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(["분기", "산업코드", "산업명", "명목_십억원", "실질_십억원"])
        for code, nm in leaves:
            n, rl = nom.get(code, {}), real.get(code, {})
            for t in sorted(set(n) | set(rl)):
                w.writerow([t, code, nm, n.get(t, ""), rl.get(t, "")])

    # 산업별 커버리지
    with open("gdp_industry_coverage.csv", "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(["산업코드", "산업명", "명목_시작", "명목_종료", "명목_개수",
                    "실질_시작", "실질_종료", "실질_개수"])
        for code, nm in leaves:
            n, rl = sorted(nom.get(code, {})), sorted(real.get(code, {}))
            w.writerow([code, nm,
                        n[0] if n else "", n[-1] if n else "", len(n),
                        rl[0] if rl else "", rl[-1] if rl else "", len(rl)])

    print(f"산업 {len(leaves)}개 저장 완료")
    print(f"{'산업명':<28}{'명목구간':>22}{'실질구간':>22}")
    for code, nm in leaves:
        n, rl = sorted(nom.get(code, {})), sorted(real.get(code, {}))
        ns = f"{n[0]}~{n[-1]}({len(n)})" if n else "없음"
        rs = f"{rl[0]}~{rl[-1]}({len(rl)})" if rl else "없음"
        print(f"{nm[:26]:<28}{ns:>22}{rs:>22}")


if __name__ == "__main__":
    main()


## 3. fetch_io_tables.py — 산업연관표 수집

In [ ]:
# -*- coding: utf-8 -*-
"""한국은행 산업연관표(I-O) 계수행렬을 전 구간 받아온다 (대분류, 2015~2023).

VAR 계수의 prior 정보로 쓰기 위한 산업 간 연계강도 행렬:
  - 투입계수표      : a_ij = 산업 j 한 단위 산출에 직접 투입되는 산업 i 산출 (직접 연계)
  - 생산유발계수표  : (I-A)^-1, 산업 j 최종수요 1단위가 유발하는 산업 i 총산출 (직·간접)

빈티지(기준년)별로 코드가 다름:
  생산유발계수: 271Y010(2015기준,'15~'19) + 271Y112(2020기준,'20~'23)
  투입계수    : 271Y009(2015기준,'15~'19) + 271Y111(2020기준,'20~'23)
ITEM_CODE1/NAME1 = 행(유발되는·투입하는 산업 i), ITEM_CODE2/NAME2 = 열(수요·산출 산업 j).

산출:
  io_생산유발계수_long.csv, io_투입계수_long.csv  (연도,행코드,행산업,열코드,열산업,값,기준년)
  io_sectors.csv  (대분류 산업 목록)
"""
import csv
import requests

API_KEY = open("API_KEY_BOK.txt", encoding="utf-8").read().strip()
URL = "http://ecos.bok.or.kr/api/StatisticSearch/{k}/json/kr/1/100000/{t}/A/1900/2099"
TABLES = {
    "생산유발계수": [("271Y010", "2015기준"), ("271Y112", "2020기준")],
    "투입계수":     [("271Y009", "2015기준"), ("271Y111", "2020기준")],
}


def fetch(table):
    r = requests.get(URL.format(k=API_KEY, t=table), timeout=120).json()
    return r.get("StatisticSearch", {}).get("row", []) or []


def main():
    sectors = {}
    for kind, vintages in TABLES.items():
        out = []
        for code, base in vintages:
            for r in fetch(code):
                v = r.get("DATA_VALUE")
                if v in (None, "", "-"):
                    continue
                out.append([r["TIME"], r["ITEM_CODE1"], r["ITEM_NAME1"],
                            r["ITEM_CODE2"], r["ITEM_NAME2"], float(v), base])
                sectors[r["ITEM_CODE1"]] = r["ITEM_NAME1"]
        fn = f"io_{kind}_long.csv"
        with open(fn, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.writer(f)
            w.writerow(["연도", "행코드_i", "행산업_i", "열코드_j", "열산업_j", "값", "기준년"])
            w.writerows(out)
        yrs = sorted(set(r[0] for r in out))
        print(f"{kind}: {len(out)}행, {yrs[0]}~{yrs[-1]} ({len(yrs)}개년), "
              f"산업 {len(set(r[1] for r in out))}개 → {fn}")

    with open("io_sectors.csv", "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(["코드", "산업명"])
        for c in sorted(sectors):
            w.writerow([c, sectors[c]])
    print(f"대분류 산업 {len(sectors)}개 → io_sectors.csv")


if __name__ == "__main__":
    main()


## 4. fetch_exog.py — 거시 외생변수 수집·정상성 변환

In [ ]:
# -*- coding: utf-8 -*-
"""VARX 외생변수 구축 — 정상성 변환된 거시 외생 블록 (분기).

변수와 변환 (모두 정상성 지향):
  유가     oil_g   : 100·Δlog(Brent, USD/bbl, 분기평균)        [FRED MCOILBRENTEU]
  노동     lab_g   : 100·Δlog(취업자수, 분기평균)              [ECOS 901Y027 I61BA]
  교역량   trd_g   : 100·Δlog((수출물량+수입물량)/2, 분기평균) [ECOS 403Y002/403Y004 *AA]
  실질금리 rrate   : 회사채(3년,AA-) − CPI 전년동기 인플레이션  [ECOS 721Y001 7020000, 901Y009]
  실질환율 rfx_g   : 100·Δlog(실질 원/달러),
                     실질원달러 = 명목(원/달러) × 미국CPI / 한국CPI [731Y006, 902Y008, 901Y009]

월자료는 분기평균으로 집계. 정상성은 ADF로 검정해 레벨/차분을 선택.
산출: exog_quarterly.csv (분기 x 변수), exog_coverage_adf.csv
"""
import io
import numpy as np
import pandas as pd
import requests
from statsmodels.tsa.stattools import adfuller

KEY = open("API_KEY_BOK.txt", encoding="utf-8").read().strip()
BASE = "http://ecos.bok.or.kr/api/StatisticSearch/{k}/json/kr/1/100000/{t}/M/190001/209912/{it}"


def ecos_m(table, *items):
    """월자료 → pandas Series (index=Timestamp, 분기말 아님)."""
    url = BASE.format(k=KEY, t=table, it="/".join(items))
    rows = requests.get(url, timeout=120).json().get("StatisticSearch", {}).get("row", [])
    s = {}
    for r in rows:
        v = r.get("DATA_VALUE")
        if v in (None, "", "-"):
            continue
        t = r["TIME"]
        s[pd.Timestamp(int(t[:4]), int(t[4:6]), 1)] = float(v)
    return pd.Series(s).sort_index()


def to_q(s):
    """월 → 분기평균, index를 'YYYYQq' 문자열로."""
    q = s.resample("QE").mean()
    return pd.Series(q.values, index=[f"{d.year}Q{(d.month-1)//3+1}" for d in q.index])


def fred_brent():
    txt = requests.get(
        "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MCOILBRENTEU", timeout=30).text
    df = pd.read_csv(io.StringIO(txt))
    df.columns = ["date", "v"]
    df["date"] = pd.to_datetime(df["date"])
    df = df[df["v"] != "."]
    return pd.Series(df["v"].astype(float).values,
                     index=df["date"]).sort_index()


def qspan(start="1980Q1", end="2025Q4"):
    return [f"{y}Q{q}" for y in range(int(start[:4]), int(end[:4]) + 1) for q in range(1, 5)
            if f"{y}Q{q}" >= start and f"{y}Q{q}" <= end]


def adf_p(x):
    x = x.dropna()
    return adfuller(x, autolag="AIC")[1] if len(x) > 12 else np.nan


def main():
    # 원자료(분기) 적재
    oil = to_q(fred_brent())                                   # Brent USD
    lab = to_q(ecos_m("901Y027", "I61BA"))                     # 취업자수
    xq = to_q(ecos_m("403Y002", "*AA"))                        # 수출물량
    mq = to_q(ecos_m("403Y004", "*AA"))                        # 수입물량
    trd = (xq + mq) / 2                                        # 교역량(평균)
    nrate = to_q(ecos_m("721Y001", "7020000"))                # 회사채3년 명목
    krcpi = to_q(ecos_m("901Y009", "0"))                      # 한국 CPI
    uscpi = to_q(ecos_m("902Y008", "US"))                     # 미국 CPI
    nfx = to_q(ecos_m("731Y006", "0000003", "0000100"))       # 명목 원/달러(월평균)

    idx = qspan()
    R = lambda s: s.reindex(idx)

    # 변환
    g = lambda s: 100 * np.log(R(s)).diff()                    # 100·Δlog
    cpi_yoy = 100 * (R(krcpi) / R(krcpi).shift(4) - 1)         # 전년동기 인플레이션
    rrate = R(nrate) - cpi_yoy                                 # 실질금리(ex-post)
    rfx = R(nfx) * R(uscpi) / R(krcpi)                         # 실질 원/달러
    rfx_g = 100 * np.log(rfx).diff()                           # 실질환율 변화율

    out = pd.DataFrame({
        "oil_g": g(oil), "lab_g": g(lab), "trd_g": g(trd),
        "d_rrate": rrate.diff(),        # 실질금리 변화 (레벨이 비정상이라 차분)
        "rfx_g": rfx_g,                 # 실질환율 변화율
        "rrate_lvl": rrate, "rfx_lvl": np.log(rfx) * 100,   # 참고용(비정상)
    }, index=idx)

    # 커버리지 + ADF (레벨/차분 후보 모두)
    cov = []
    for c in out.columns:
        s = out[c].dropna()
        cov.append([c, s.index[0] if len(s) else "", s.index[-1] if len(s) else "",
                    len(s), round(adf_p(out[c]), 4)])
    cov = pd.DataFrame(cov, columns=["변수", "시작", "종료", "관측수", "ADF_p(레벨/현재변환)"])

    out.to_csv("exog_quarterly.csv", encoding="utf-8-sig")
    cov.to_csv("exog_coverage_adf.csv", index=False, encoding="utf-8-sig")

    print(cov.to_string(index=False))
    # 공통표본(노동 포함 / 제외)
    core = ["oil_g", "trd_g", "d_rrate", "rfx_g"]
    allv = core + ["lab_g"]
    cs_all = out[allv].dropna().index
    cs_core = out[core].dropna().index
    print(f"\n공통표본(5변수, 노동포함): {cs_all[0]} ~ {cs_all[-1]}  ({len(cs_all)}분기)")
    print(f"공통표본(4변수, 노동제외): {cs_core[0]} ~ {cs_core[-1]}  ({len(cs_core)}분기)")
    print("저장: exog_quarterly.csv, exog_coverage_adf.csv")


if __name__ == "__main__":
    main()


## 5. backcast_labor.py — 취업자 1990년대 칼만평활 복원

In [ ]:
# -*- coding: utf-8 -*-
"""취업자 증가율(lab_g) 백캐스팅 — 동적요인 칼만평활 + 홀드아웃 검증.

lab_g(1999Q3~)를 1990Q3까지 복원. 두 예측자 집합을 비교:
  (A) 외생 4종만        : oil_g, trd_g, d_rrate, rfx_g
  (B) + 실질GDP 증가율  : gdp_g 추가 (Okun의 법칙 — 고용의 본질적 동인)
홀드아웃(최근 관측구간을 일부러 결측처리→복원→실측 비교)으로 예측력을 정직하게 평가.
최종 백캐스트는 검증 우수안으로 1990Q3~1999Q2 결측을 복원.

산출: exog_quarterly_filled.csv, _labor_backcast.json
"""
import json
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor

SPAN0, SPAN1 = "1990Q3", "2025Q4"
BASE = ["oil_g", "trd_g", "d_rrate", "rfx_g"]


def qkey(q):
    return (int(q[:4]), int(q[5]))


def load_panel():
    ex = pd.read_csv("exog_quarterly.csv", index_col=0, encoding="utf-8-sig")
    g = pd.read_csv("gdp_quarterly_sa.csv", encoding="utf-8-sig")          # 실질 GDP
    g = g.set_index("분기")["실질GDP_십억원"].astype(float)
    gdp_g = 100 * np.log(g).diff()
    idx = sorted([q for q in ex.index if SPAN0 <= q <= SPAN1], key=qkey)
    P = ex.loc[idx, BASE + ["lab_g"]].astype(float)
    P["gdp_g"] = gdp_g.reindex(idx)
    # 생산연령인구 증가율(통계청, 연→분기 broadcast) — 인구를 노동 복원 예측자로 반영.
    # 1990년대 생산연령인구는 연 +1.5% 안팎으로 늘어, 그 시기 고용 추세의 핵심 닻이 된다.
    pop = pd.read_csv("kosis_wap_historical.csv", encoding="utf-8-sig").set_index("연도")["생산연령인구_천명"]
    wap = 100 * np.log(pop).diff()
    P["wap_g"] = [wap.get(int(q[:4]), np.nan) for q in idx]
    return P, idx


def smooth_labor(panel, cols):
    """동적요인모형 적합 후 lab_g 평활 재구성 (표준화 역변환), (lab_hat, se)."""
    Z0 = panel[cols + ["lab_g"]]
    mu, sd = Z0.mean(), Z0.std()
    Z = (Z0 - mu) / sd
    mod = DynamicFactor(Z, k_factors=2, factor_order=2, error_order=1,
                        enforce_stationarity=True)
    res = mod.fit(disp=False, maxiter=400, method="lbfgs")
    d = res.filter_results.design
    d = d[:, :, 0] if d.ndim == 3 else d
    li = (cols + ["lab_g"]).index("lab_g")
    a, Pc = res.smoothed_state, res.smoothed_state_cov
    zr = d[li]
    yhat = zr @ a
    var = np.clip([zr @ Pc[:, :, t] @ zr for t in range(Pc.shape[2])], 0, None)
    return yhat * sd["lab_g"] + mu["lab_g"], np.sqrt(var) * sd["lab_g"]


def holdout(panel, cols, test=("2019Q1", "2021Q4")):
    """관측구간 일부를 결측처리→복원→실측 비교 (블록 백캐스트 모사)."""
    idx = list(panel.index)
    mask = [(test[0] <= q <= test[1]) for q in idx]
    actual = panel["lab_g"].values.copy()
    pp = panel.copy()
    pp.loc[[q for q, m in zip(idx, mask) if m], "lab_g"] = np.nan
    hat, _ = smooth_labor(pp, cols)
    m = np.array(mask) & ~np.isnan(actual)
    rmse = np.sqrt(np.nanmean((actual[m] - hat[m]) ** 2))
    corr = np.corrcoef(actual[m], hat[m])[0, 1]
    return rmse, corr


def main():
    panel, idx = load_panel()
    obs = panel["lab_g"].values
    missing = np.isnan(obs)

    cands = {"A 외생4종": BASE,
             "B +실질GDP": BASE + ["gdp_g"],
             "C +실질GDP+생산연령인구": BASE + ["gdp_g", "wap_g"]}
    print("[홀드아웃 검증] 2019Q1~2021Q4(COVID 변동기) 결측처리 후 복원 vs 실측")
    for nm, cols in cands.items():
        r, c = holdout(panel, cols)
        print(f"  ({nm:22}): RMSE={r:.3f}  corr={c:+.3f}")
    better = cands["C +실질GDP+생산연령인구"]   # 인구 반영 (요청) — 생산연령인구 포함
    print("  → 채택: C (산출 + 생산연령인구 결합)")

    lab_hat, lab_se = smooth_labor(panel, better)
    lab_filled = np.where(missing, lab_hat, obs)

    out = panel[BASE].copy()
    out["lab_g"] = lab_filled
    out.index.name = "분기"
    out.to_csv("exog_quarterly_filled.csv", encoding="utf-8-sig")

    viz = {"q": idx,
           "obs": [None if m else round(float(v), 3) for m, v in zip(missing, obs)],
           "hat": [round(float(v), 3) for v in lab_hat],
           "lo": [round(float(h - 1.64 * s), 3) for h, s in zip(lab_hat, lab_se)],
           "hi": [round(float(h + 1.64 * s), 3) for h, s in zip(lab_hat, lab_se)],
           "adopt": "C(+GDP+생산연령인구)"}
    json.dump(viz, open("_labor_backcast.json", "w", encoding="utf-8"), ensure_ascii=False)

    print(f"\n최종 백캐스트: 결측 {int(missing.sum())}분기 복원 ({idx[0]}~1999Q2)")
    for q in ["1997Q4", "1998Q1", "1998Q2", "1998Q4", "1999Q1"]:
        i = idx.index(q)
        print(f"  lab_g {q}: {lab_filled[i]:+.2f}%  [90%: {viz['lo'][i]:+.2f}~{viz['hi'][i]:+.2f}] (백캐스트)")
    print("저장: exog_quarterly_filled.csv")


if __name__ == "__main__":
    main()
